# 01_pipeline
Captured 2026-11-21 21:37 UTC

**Raw files to governed gold.** Generator wrote 47748 PO lines with 954 planted duplicates, 1441 lines in grams and 348 negative forecasts; the pipeline quarantined the duplicates and negatives, filled missing lead times from the source list and matched every supplier quality code to an ERP vendor.


In [1]:
-- generator run summary (run_log/generate_base.json)
-- src/data/generate.py

{
 "task": "generate",
 "scenario": "base",
 "as_of": "2026-11-20",
 "seed": 42,
 "vendors": 60,
 "materials": 120,
 "source_list": 239,
 "single_source_api_pct": 37.5,
 "skus": 297,
 "bom_lines": 1035,
 "po_lines": 47748,
 "po_duplicates_planted": 954,
 "po_lines_in_grams": 1441,
 "po_lines_no_lead_time": 2980,
 "inventory_rows": 41850,
 "quality_events": 377,
 "forecast_files": 9,
 "negative_forecasts": 348
}

In [2]:
-- row counts and data quality by layer
SELECT ... FROM bronze/silver/gold

t                            | n    
-------------------------------------------------
silver.po_line lead time fil | 2980 
silver.supplier_event unmatc | 0    
gold.api_exposure            | 288  
gold.days_of_cover           | 288  
gold.disruption_impact       | 38   
bronze.po_line               | 48702
silver.po_line               | 47748
silver.po_line_quarantine (d | 954  
silver.forecast_quarantine ( | 42   
(9 rows)

In [3]:
-- supplier code conformance
SELECT match_method, count(*) FROM supply_chain.silver.supplier_xref GROUP BY 1

match_method | n 
--------------------
DUNS         | 60
(1 rows)

In [4]:
-- file-arrival trigger: a new supplier quality file starts the job by itself
databricks jobs list-runs --job-id <radar_main>

[
 {
  "run_id": 588540134005639,
  "trigger": "FILE_ARRIVAL",
  "result": "SUCCESS",
  "tasks": [
   "explain_impact",
   "generate",
   "pipeline",
   "refresh_gold",
   "score_po_risk",
   "should_generate",
   "sync_lakebase"
  ]
 }
]

In [5]:
-- latest forecast file wins (late and re-sent files ignored)
SELECT published_at, count(*) FROM silver.forecast GROUP BY 1

published_at | n   
----------------------
2026-11-20   | 7454
2026-11-13   | 288 
2026-11-06   | 288 
2026-10-30   | 288 
(4 rows)